# K Nearest Neighbours and Support Vector Machines

Both models draw a boundary between classes. KNN votes among the nearest points; SVM chooses the widest margin. Both need scaled features because they depend on distance.

## Imports and loading helper

In [ ]:
# Standard imports for the whole course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Find the Datasets folder whether the notebook is run from the
# project root or from inside the notebooks folder.
from pathlib import Path
for candidate in [Path("Datasets"), Path("../Datasets")]:
    if candidate.exists():
        DATA = candidate
        break

def load(name):
    """Load a CSV from the Datasets folder by file name."""
    return pd.read_csv(DATA / name)

print("Datasets are in:", DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

## Preparing the customer data

In [ ]:
df = load("Naive_Bayes_Decision_RF_Tree_Boosting_User_Data.csv")
X = df.iloc[:, [2, 3]].values   # age and estimated salary
y = df.iloc[:, 4].values        # purchase outcome

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## K Nearest Neighbours

Choose k, measure distances, and let the nearest neighbours vote. The default distance is Euclidean.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2)
knn.fit(X_train, y_train)
print("KNN accuracy:", accuracy_score(y_test, knn.predict(X_test)))

## Support Vector Machine

A linear kernel finds a straight boundary with the widest possible margin.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="linear", random_state=0)
svm.fit(X_train, y_train)
print("SVM accuracy:", accuracy_score(y_test, svm.predict(X_test)))

## Drawing the decision boundaries

We paint the plane with the predicted class to show where each model puts the split.

In [ ]:
def plot_boundary(model, title):
    x1 = np.linspace(X_train[:, 0].min() - 0.5,
                     X_train[:, 0].max() + 0.5, 300)
    x2 = np.linspace(X_train[:, 1].min() - 0.5,
                     X_train[:, 1].max() + 0.5, 300)
    xx, yy = np.meshgrid(x1, x2)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
                color="red", s=14, label="not purchased")
    plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
                color="green", s=14, label="purchased")
    plt.title(title); plt.legend(); plt.show()

plot_boundary(knn, "KNN decision boundary")
plot_boundary(svm, "Linear SVM decision boundary")